In [1]:
%cd ../../..

/Users/hoangle/Projects/untangling-people/Food-Waste-Optimization


In [2]:
from pathlib import Path

import pandas as pd
import psycopg as pg
from loguru import logger
from psycopg import sql
from psycopg.rows import dict_row

In [3]:
USER = ""
PWD = ""
PORT = ""
HOST = ""
DB_NAME = ""

# Upload table `menus`

In [4]:
path_dir = Path("data/processed/phase_4/menus")

list_df = [pd.read_parquet(path) for path in path_dir.glob("*.parquet")]
df = pd.concat(list_df)

df.head()

,date,restaurant,index,meal_ids,fitness
0,2025-06-17,phy,3853,"[2205, 9105, 710]",1.753116
1,2025-06-17,phy,805,"[3010, 9106, 6142]",1.758633
2,2025-06-17,phy,4452,"[6137, 20017, 1446]",7.398146
3,2025-06-16,phy,403,"[6355, 9105, 2943]",1.735333
4,2025-06-20,phy,1906,"[2207, 9106, 2655]",1.790416


In [5]:
values = []

for r in df.itertuples():
    row = (r.index, r.date, r.restaurant, r.meal_ids.tolist(), r.fitness)

    values.append(row)

In [7]:
try:
    with pg.connect(
        user=USER,
        password=PWD,
        host=HOST,
        port=PORT,
        dbname=DB_NAME,
        row_factory=dict_row,
    ) as conn:
        with conn.cursor() as cur:

            # Compose SQL
            cols = ['index', 'date', 'restaurant', 'meal_ids', 'fitness']
            table = 'menu'

            query = """
                insert into {table}
                    ({cols})
                values ({values})
                ;
            """

            stmt = (
                sql
                .SQL(query)
                .format(
                    table=sql.Identifier(table),
                    cols=sql.SQL(', ').join(map(sql.Identifier, cols)),
                    values=sql.SQL(', ').join(sql.Placeholder() * len(values[0]))
                )
            )

            # for v in values:
            cur.executemany(stmt, values)
            conn.commit()

            # ret = cur.fetchall()
            

except pg.OperationalError as e:
    logger.error(f"Connect to DB got error: {e}")

# Upload table `meals`

In [4]:
path = "data/processed/phase_4/dim_meals.parquet"
dim_meals_raw = pd.read_parquet(path)
dim_meals_raw.head()

,meal_id,meal_type,schoolyear,restaurant,attributes,aliases
0,9017,vegan,24-25,"[che, exa, vik]","[vegan-miscellaneous, kela]","[""Butter"" härkäpapua & pähkinää]"
1,7201,vegan,23-24,None,[],[2023 Härkäpu-sienilasagnette]
2,9032,vegan,23-24,None,[],[Appelisiini-luomukikhernecurrya]
3,9102,vegan,23-24,None,[],[Artisokkavugetteja & tuoretomaattisalsaa]
4,7010,vegetarian,24-25,"[che, exa, vik]",[],"[Aurajuusto-pinaattilasagnette, Aurajuusto-pin..."


In [11]:
def _tolist(x):
    if x is not None:
        return x.tolist()
    return x

values = [
    (r.meal_id, r.meal_type, r.schoolyear, _tolist(r.restaurant), _tolist(r.attributes), _tolist(r.aliases),)
    for r in dim_meals_raw.itertuples()
]

In [13]:
# Compose SQL
cols = dim_meals_raw.columns
table = 'meals'

try:
    with pg.connect(
        user=USER,
        password=PWD,
        host=HOST,
        port=PORT,
        dbname=DB_NAME,
        row_factory=dict_row,
    ) as conn:
        with conn.cursor() as cur:
            query = """
                insert into {table}
                    ({cols})
                values ({values})
                ;
            """

            stmt = (
                sql
                .SQL(query)
                .format(
                    table=sql.Identifier(table),
                    cols=sql.SQL(', ').join(map(sql.Identifier, cols)),
                    values=sql.SQL(', ').join(sql.Placeholder() * len(values[0]))
                )
            )

            # for v in values:
            cur.executemany(stmt, values)
            conn.commit()

            # ret = cur.fetchall()
            

except pg.OperationalError as e:
    logger.error(f"Connect to DB got error: {e}")

# Upload table `biowaste`

In [8]:
path = "data/processed/phase_4/dim_waste.xlsx"
dim_waste_raw = pd.read_excel(path)
dim_waste_raw.head()

,meal_id,waste
0,9017,0.010000
1,7201,0.010000
2,9032,0.010000
3,9102,0.010000
4,7010,0.039256


In [9]:
values = [
    (r.meal_id, r.waste)
    for r in dim_waste_raw.itertuples()
]

In [10]:
# Compose SQL
cols = dim_waste_raw.columns
table = 'biowaste'

try:
    with pg.connect(
        user=USER,
        password=PWD,
        host=HOST,
        port=PORT,
        dbname=DB_NAME,
        row_factory=dict_row,
    ) as conn:
        with conn.cursor() as cur:
            query = """
                insert into {table}
                    ({cols})
                values ({values})
                ;
            """

            stmt = (
                sql
                .SQL(query)
                .format(
                    table=sql.Identifier(table),
                    cols=sql.SQL(', ').join(map(sql.Identifier, cols)),
                    values=sql.SQL(', ').join(sql.Placeholder() * len(values[0]))
                )
            )

            # for v in values:
            cur.executemany(stmt, values)
            conn.commit()

            # ret = cur.fetchall()
            

except pg.OperationalError as e:
    logger.error(f"Connect to DB got error: {e}")

# Upload table `co2`

In [11]:
path = "data/processed/phase_4/dim_co2.xlsx"
dim_co2_raw = pd.read_excel(path)
dim_co2_raw.head()

,meal_id,co2
0,34,0.81
1,37,0.61
2,710,0.67
3,713,0.56
4,724,0.82


In [12]:
values = [
    (r.meal_id, r.co2)
    for r in dim_co2_raw.itertuples()
]

In [13]:
# Compose SQL
cols = dim_co2_raw.columns
table = 'co2'

try:
    with pg.connect(
        user=USER,
        password=PWD,
        host=HOST,
        port=PORT,
        dbname=DB_NAME,
        row_factory=dict_row,
    ) as conn:
        with conn.cursor() as cur:
            query = """
                insert into {table}
                    ({cols})
                values ({values})
                ;
            """

            stmt = (
                sql
                .SQL(query)
                .format(
                    table=sql.Identifier(table),
                    cols=sql.SQL(', ').join(map(sql.Identifier, cols)),
                    values=sql.SQL(', ').join(sql.Placeholder() * len(values[0]))
                )
            )

            cur.executemany(stmt, values)
            conn.commit()
            

except pg.OperationalError as e:
    logger.error(f"Connect to DB got error: {e}")

# Upload table `pieces_whole`

In [14]:
path = "data/processed/phase_4/dim_pieces_whole.xlsx"
dim_pieces_whole_raw = pd.read_excel(path)
dim_pieces_whole_raw.head()

,date,pcs,restaurant
0,2024-11-01,151.72,phy
1,2024-11-04,235.53,phy
2,2024-11-05,248.56,phy
3,2024-11-06,257.29,phy
4,2024-11-07,262.31,phy


In [15]:
values = [
    (r.date, r.pcs, r.restaurant)
    for r in dim_pieces_whole_raw.itertuples()
]

In [16]:
# Compose SQL
cols = dim_pieces_whole_raw.columns
table = 'pieces_whole'

try:
    with pg.connect(
        user=USER,
        password=PWD,
        host=HOST,
        port=PORT,
        dbname=DB_NAME,
        row_factory=dict_row,
    ) as conn:
        with conn.cursor() as cur:
            query = """
                insert into {table}
                    ({cols})
                values ({values})
                ;
            """

            stmt = (
                sql
                .SQL(query)
                .format(
                    table=sql.Identifier(table),
                    cols=sql.SQL(', ').join(map(sql.Identifier, cols)),
                    values=sql.SQL(', ').join(sql.Placeholder() * len(values[0]))
                )
            )

            cur.executemany(stmt, values)
            conn.commit()
            

except pg.OperationalError as e:
    logger.error(f"Connect to DB got error: {e}")